In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

import joblib


In [12]:
class DatasetMF:
    def __init__(self, excel_path, targets):
        self.df = pd.read_excel(excel_path)
        self.targets = targets

    def split_train_test(self, test_size=0.2, random_state=42):
        self.X = self.df.drop(columns=self.targets + ["Player"])
        self.y = self.df[self.targets]

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state
        )
        return self

    def get_player_last_row(self, player_name):
        df_player = self.df[self.df["Player"] == player_name]
        if df_player.empty:
            raise ValueError(f"Joueur '{player_name}' introuvable")
        return df_player.iloc[-1:]

    def available_test_players(self):
        return sorted(self.df.loc[self.X_test.index, "Player"].unique())


In [13]:
class ModelMF:
    def __init__(self, model):
        self.model = model
        self.scaler = StandardScaler()

    def train(self, X_train, y_train):
        X_scaled = self.scaler.fit_transform(X_train)
        self.model.fit(X_scaled, y_train)

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def evaluate(self, X_test, y_test):
        preds = self.predict(X_test)
        return {
            "MAE": mean_absolute_error(y_test, preds),
            "R2": r2_score(y_test, preds)
        }


In [14]:
class ModelTrainerMF:
    def __init__(self, dataset):
        self.dataset = dataset
        self.models = {}
        self.scores = []

    def model_defs(self):
        return {
            "LinearRegression": LinearRegression(),
            "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
            "GradientBoosting": GradientBoostingRegressor(random_state=42),
            "SVR": SVR()
        }

    def train(self):
        for target in self.dataset.targets:
            self.models[target] = {}

            for name, base_model in self.model_defs().items():
                model = ModelMF(base_model)
                model.train(self.dataset.X_train, self.dataset.y_train[target])

                metrics = model.evaluate(
                    self.dataset.X_test,
                    self.dataset.y_test[target]
                )

                self.models[target][name] = model

                self.scores.append({
                    "Target": target,
                    "Model": name,
                    "MAE": metrics["MAE"],
                    "R2": metrics["R2"]
                })

        return pd.DataFrame(self.scores).sort_values(["Target", "MAE"])


In [15]:
class PlayerPredictorMF:
    def __init__(self, dataset, trainer):
        self.dataset = dataset
        self.trainer = trainer

    def predict_player(self, player_name):
        row = self.dataset.get_player_last_row(player_name)
        X = row.drop(columns=self.dataset.targets + ["Player"])

        results = {}
        for target, models in self.trainer.models.items():
            results[target] = {}
            for model_name, model in models.items():
                results[target][model_name] = round(float(model.predict(X)[0]), 3)

        return results


In [ ]:
def compare_real_vs_predicted_mf(dataset, predictor, player_name):
    last_row = dataset.get_player_last_row(player_name)
    real = last_row[dataset.targets].iloc[0]

    preds = predictor.predict_player(player_name)

    rows = []
    for target in dataset.targets:
        for model, value in preds[target].items():
            rows.append({
                "Target": target,
                "Model": model,
                "Real": round(real[target], 3),
                "Predicted": value,
                "Error": round(value - real[target], 3)
            })

    return pd.DataFrame(rows)


In [ ]:
EXCEL_PATH_MF = r"C:\Users\Aref Bakali\OneDrive\Bureau\Projet Python\data\selection\features_MF_selected.xlsx"
TARGETS_MF = [
    "Total Cmp%",
    "PrgC/90",
    "PrgP/90",
    "Chllngs Lost/90",
    "KP/90"
]


In [18]:
dataset_mf = DatasetMF(EXCEL_PATH_MF, TARGETS_MF)
dataset_mf.split_train_test()

trainer_mf = ModelTrainerMF(dataset_mf)
results_mf = trainer_mf.train()

results_mf


,Target,Model,MAE,R2
15,Chllngs Lost/90,SVR,0.122506,0.846599
14,Chllngs Lost/90,GradientBoosting,0.152648,0.811495
13,Chllngs Lost/90,RandomForest,0.168641,0.770354
12,Chllngs Lost/90,LinearRegression,0.177386,0.729105
19,KP/90,SVR,0.148617,0.839016
16,KP/90,LinearRegression,0.173442,0.827127
18,KP/90,GradientBoosting,0.174455,0.826837
17,KP/90,RandomForest,0.186359,0.805625
7,PrgC/90,SVR,0.228628,0.860461
6,PrgC/90,GradientBoosting,0.279624,0.822093


In [19]:
dataset_mf.available_test_players()[:20]


['Abdou Harroui',
 'Abdoulaye Touré',
 'Adam Lallana',
 'Adrien Rabiot',
 'Adrien Tameze',
 'Adrien Thomasson',
 'Agustín Martegani',
 'Albin Ekdal',
 'Aleix García',
 'Aleksandr Golovin',
 'Aleksei Miranchuk',
 'Alex Král',
 'Alex Scott',
 'Alex Sola',
 'Alexis Blin',
 'Alfred Duncan',
 'Aljoscha Kemlein',
 'Amadou Haidara',
 'Amir Richardson',
 'Andre-Frank Zambo Anguissa']

In [22]:
predictor_mf = PlayerPredictorMF(dataset_mf, trainer_mf)

player_name = dataset_mf.available_test_players()[5]
player_name
pd.DataFrame(predictor_mf.predict_player(player_name)).T
compare_real_vs_predicted_mf(dataset_mf, predictor_mf, player_name)


,Target,Model,Real,Predicted,Error
0,Total Cmp%,LinearRegression,81.800,82.319,0.519
1,Total Cmp%,RandomForest,81.800,81.203,-0.597
2,Total Cmp%,GradientBoosting,81.800,82.315,0.515
3,Total Cmp%,SVR,81.800,81.894,0.094
4,PrgC/90,LinearRegression,1.031,0.181,-0.850
5,PrgC/90,RandomForest,1.031,1.092,0.061
6,PrgC/90,GradientBoosting,1.031,1.056,0.025
7,PrgC/90,SVR,1.031,1.205,0.174
8,PrgP/90,LinearRegression,5.120,3.995,-1.125
9,PrgP/90,RandomForest,5.120,5.479,0.359


In [24]:
def top_10_mf_all_targets(dataset, trainer):
    rows = []

    for target, models in trainer.models.items():
        best_model = min(
            models,
            key=lambda m: mean_absolute_error(
                dataset.y_test[target],
                models[m].predict(dataset.X_test)
            )
        )

        model = models[best_model]
        preds = model.predict(dataset.X)

        df_tmp = pd.DataFrame({
            "Player": dataset.df["Player"],
            "Predicted": preds
        }).sort_values("Predicted", ascending=False).head(10)

        df_tmp["Target"] = target
        df_tmp["Model"] = best_model

        rows.append(df_tmp)

    return pd.concat(rows)
top_10_mf_all_targets(dataset_mf, trainer_mf)



,Player,Predicted,Target,Model
317,Stanislav Lobotka,95.935099,Total Cmp%,LinearRegression
559,Marco Verratti,95.114649,Total Cmp%,LinearRegression
1021,Rodri,94.798728,Total Cmp%,LinearRegression
1383,Pierre Højbjerg,94.371067,Total Cmp%,LinearRegression
1402,Frenkie de Jong,94.194706,Total Cmp%,LinearRegression
569,Vitinha,93.999599,Total Cmp%,LinearRegression
1243,Dani Ceballos,93.958512,Total Cmp%,LinearRegression
531,Aurélien Tchouaméni,93.861654,Total Cmp%,LinearRegression
1128,Granit Xhaka,93.833466,Total Cmp%,LinearRegression
857,Mateo Kovačić,93.733648,Total Cmp%,LinearRegression


In [ ]:
from sklearn.metrics import mean_absolute_error
import joblib
import os

# 📂 Dossier où tu stockes TOUS les modèles
SAVE_DIR = r"C:\Users\Aref Bakali\OneDrive\Bureau\Projet Python\ML_Notebooks"
os.makedirs(SAVE_DIR, exist_ok=True)

for target, models in trainer_mf.models.items():
    # 🔍 Sélection du meilleur modèle (MAE minimal)
    best_model_name = min(
        models,
        key=lambda m: mean_absolute_error(
            dataset_mf.y_test[target],
            models[m].predict(dataset_mf.X_test)
        )
    )

    best_model = models[best_model_name]

    # 💾 Sauvegarde scaler + modèle
    joblib.dump(
        {
            "model_name": best_model_name,
            "scaler": best_model.scaler,
            "model": best_model.model
        },
        os.path.join(
            SAVE_DIR,
            f"best_MF_{target.replace('/', '_').replace(' ', '_')}.pkl"
        )
    )

    print(f"✅ Modèle MF sauvegardé : best_MF_{target}→ {best_model_name}")


✅ Modèle MF sauvegardé : best_MF_Total Cmp%
✅ Modèle MF sauvegardé : best_MF_PrgC/90
✅ Modèle MF sauvegardé : best_MF_PrgP/90
✅ Modèle MF sauvegardé : best_MF_Chllngs Lost/90
✅ Modèle MF sauvegardé : best_MF_KP/90
